# Zeta and Character N-grams

**Computational Text Analysis Pack · Module 09 — Stylometry**

*Technique:* Craig's Zeta on mid-frequency words; character n-gram attribution

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/07-stylometry/09e_zeta_and_character_ngrams.ipynb)

Data: `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## What this notebook does

Burrows's Delta works on the *most frequent* words — function words, used
unconsciously. Two other feature sets in standard use look elsewhere.

**Craig's Zeta** deliberately ignores both ends of the frequency range and works
on mid-frequency vocabulary, scored by how consistently an author *uses* a word
rather than how often. **Character n-grams** ignore words entirely and work on
letter sequences, which is why they survive OCR noise and cross language
boundaries where word tokenisation struggles.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
works = (df_colonial.groupby(["work_id", "title", "author"])["text"]
                    .apply(" ".join).reset_index())
works["tokens"] = works["text"].map(tokenise)
works = works[works["tokens"].map(len) >= 5000].reset_index(drop=True)
counts = works["author"].value_counts()
target = counts.index[0]
print(f"Comparing {target} ({counts.iloc[0]} works) against everyone else")
print(f"({len(works) - counts.iloc[0]} works by {works['author'].nunique() - 1} authors)")

## Zeta: presence, not frequency

Zeta splits both corpora into equal-sized segments and asks, for each word: in
what proportion of *this author's* segments does it appear, and in what
proportion of the *other* author's? The score is the difference.

Because it counts segments rather than tokens, one passage that repeats a word
forty times cannot dominate — which is exactly the failure mode Zeta was
designed to avoid.

In [ ]:
SEG = 2000

def segments(rows):
    out = []
    for toks in rows:
        for i in range(0, len(toks) - SEG + 1, SEG):
            out.append(set(toks[i:i + SEG]))
    return out

a_segs = segments(works.loc[works["author"] == target, "tokens"])
b_segs = segments(works.loc[works["author"] != target, "tokens"])
print(f"{len(a_segs)} segments for {target}, {len(b_segs)} for the rest")

vocab = Counter()
for s in a_segs + b_segs:
    vocab.update(s)
mid = [w for w, n in vocab.items()
       if 0.05 * len(a_segs + b_segs) < n < 0.65 * len(a_segs + b_segs)]

rows = []
for w in mid:
    pa = sum(1 for s in a_segs if w in s) / len(a_segs)
    pb = sum(1 for s in b_segs if w in s) / len(b_segs)
    rows.append({"word": w, "in_target": pa, "in_others": pb, "zeta": pa - pb})
zeta = pd.DataFrame(rows).sort_values("zeta", ascending=False)
print(f"\n{len(zeta)} mid-frequency words scored")

In [ ]:
print(f"MARKER words — {target} uses these, others largely do not:")
print(zeta.head(12).to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print()
print(f"AVOIDED words — others use these, {target} largely does not:")
print(zeta.tail(12).iloc[::-1].to_string(index=False,
                                         float_format=lambda v: f"{v:.3f}"))

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(zeta["in_others"], zeta["in_target"], s=8, alpha=.4)
plt.plot([0, 1], [0, 1], "--", color="grey", lw=1)
for _, r in pd.concat([zeta.head(6), zeta.tail(6)]).iterrows():
    plt.annotate(r["word"], (r["in_others"], r["in_target"]), fontsize=7)
plt.xlabel("proportion of OTHER authors' segments")
plt.ylabel(f"proportion of {target}'s segments")
plt.title("Craig's Zeta: distance from the diagonal is the signal")
plt.tight_layout()

## Character n-grams

No tokenisation, no vocabulary, no stopword list. Character n-grams capture
morphology, punctuation habits and spelling at once — which is why they are the
feature set of choice for noisy OCR and for languages where word boundaries are
not marked by spaces.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

multi = counts[counts >= 2].index
sub = works[works["author"].isin(multi)].reset_index(drop=True)
y = sub["author"].to_numpy()

for label, vec in [
    ("words (1-gram)", TfidfVectorizer(token_pattern=r"[\w\u0300-\u036f]+", max_features=3000)),
    ("chars (3-gram)", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 3),
                                       max_features=3000)),
    ("chars (4-gram)", TfidfVectorizer(analyzer="char_wb", ngram_range=(4, 4),
                                       max_features=3000)),
]:
    pipe = make_pipeline(vec, LogisticRegression(max_iter=2000))
    acc = cross_val_score(pipe, sub["text"], y, cv=LeaveOneOut()).mean()
    print(f"{label:<18} leave-one-out accuracy {acc:.3f}")

### 🔧 Your turn 1

Change SEG from 2000 to 500 and re-run the Zeta section. Do the marker words change character?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Short segments push proportions toward the extremes — a word has to appear in a
500-word window to count, so most words appear in few segments and the scores
compress. The marker list starts filling with moderately common words rather
than distinctive ones. Craig's original work uses segments in the low thousands
for exactly this reason, and as with MATTR the segment size must be reported:
Zeta scores are not comparable across studies that chose different ones.

</details>